# Neuro IA Marketing — CNN de expresiones faciales
TensorFlow/Keras · Entrenamiento desde cero · Seis clases, sin miedo.

En VS Code selecciona **Seleccionar kernel → Entornos de Python → .venv** (Python 3.12).
Ejecuta las celdas en orden. La sección **Entrenar** inicia el entrenamiento real.
Las imágenes se leen desde `train` y `test`, sin moverlas ni modificarlas.
El 20 % de `train` se reserva para validación; `test` solo se usa al final.
Los resultados se guardan en `modelos/CNN_fecha_hora`.

La salida estima la clase de expresión facial; no demuestra el estado emocional interno de una persona.


In [1]:
from pathlib import Path
from collections import Counter
import sys
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

# Abre este notebook desde la raíz del proyecto.
BASE = Path.cwd()
if not (BASE / 'train').is_dir() or not (BASE / 'test').is_dir():
    raise FileNotFoundError(f'Abre la carpeta Neuro-IA-Marketing en VS Code. Ruta actual: {BASE}')
CLASES = ['angry', 'disgusted', 'happy', 'neutral', 'sad', 'surprised']
SEMILLA = 42
BATCH_SIZE = 32
EPOCAS = 30
keras.utils.set_random_seed(SEMILLA)
print('Python:', sys.version.split()[0], '| TensorFlow:', tf.__version__)
print('Intérprete:', sys.executable)
print('Proyecto:', BASE)
print('GPU detectadas:', tf.config.list_physical_devices('GPU'))


Python: 3.12.14 | TensorFlow: 2.21.0
Intérprete: C:\Users\Garcia\Desktop\Neuro-IA-Marketing\.venv\Scripts\python.exe
Proyecto: C:\Users\Garcia\Desktop\Neuro-IA-Marketing
GPU detectadas: []


## 1. Separar entrenamiento y validación
La división es estratificada y reproducible. `fearful`, si existe en test, se ignora.


In [2]:
def listar_imagenes(carpeta):
    rutas, etiquetas = [], []
    for etiqueta, clase in enumerate(CLASES):
        subcarpeta = carpeta / clase
        if not subcarpeta.is_dir():
            raise FileNotFoundError(f'Falta la carpeta: {subcarpeta}')
        archivos = sorted(p for p in subcarpeta.iterdir()
                          if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg'))
        if not archivos:
            raise ValueError(f'No hay imágenes en: {subcarpeta}')
        rutas.extend(str(p) for p in archivos)
        etiquetas.extend([etiqueta] * len(archivos))
    return rutas, etiquetas

rutas, etiquetas = listar_imagenes(BASE / 'train')
x_train, x_val, y_train, y_val = train_test_split(
    rutas, etiquetas, test_size=0.20, random_state=SEMILLA, stratify=etiquetas)
x_test, y_test = listar_imagenes(BASE / 'test')
assert set(x_train).isdisjoint(x_val)
assert (set(x_train) | set(x_val)).isdisjoint(x_test)
print(f"{'Clase':<12} {'Train':>7} {'Validación':>11} {'Test':>7}")
for i, clase in enumerate(CLASES):
    print(f'{clase:<12} {Counter(y_train)[i]:>7} {Counter(y_val)[i]:>11} {Counter(y_test)[i]:>7}')
print(f'Totales: train={len(x_train)}, validación={len(x_val)}, test={len(x_test)}')


Clase          Train  Validación    Test
angry           1050         263     958
disgusted       1050         263     111
happy           1001         250    1774
neutral         1001         250    1233
sad             1001         251    1263
surprised       1001         250     831
Totales: train=6104, validación=1527, test=6170


## 2. Cargar imágenes de 48 × 48, en escala de grises
Los píxeles se normalizan al intervalo [0, 1].


In [3]:
def leer_imagen(ruta, etiqueta):
    imagen = tf.io.decode_image(tf.io.read_file(ruta), channels=1, expand_animations=False)
    imagen = tf.ensure_shape(imagen, (None, None, 1))
    imagen = tf.image.resize(imagen, (48, 48))
    return tf.cast(imagen, tf.float32) / 255.0, etiqueta

def crear_dataset(rutas, etiquetas):
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    return ds.map(leer_imagen, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = crear_dataset(x_train, y_train)
val_ds = crear_dataset(x_val, y_val)
test_ds = crear_dataset(x_test, y_test)
imagenes, etiquetas_lote = next(iter(train_ds))
print('Lote:', imagenes.shape, '| rango:', float(tf.reduce_min(imagenes)), float(tf.reduce_max(imagenes)))


Lote: (32, 48, 48, 1) | rango: 0.0 1.0


## 3. Crear la CNN
Pesos inicializados desde cero. Las transformaciones aleatorias y Dropout solo actúan al entrenar. Ejecutar esta celda de nuevo reinicia los pesos.


In [4]:
# Crear modelo: CNN pequeña, con pesos inicializados desde cero.
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(SEMILLA)
modelo = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),
    layers.RandomFlip('horizontal', seed=SEMILLA),
    layers.RandomRotation(0.04, seed=SEMILLA + 1),
], name='cnn_expresiones')

for filtros in (16, 32, 64):
    modelo.add(layers.Conv2D(filtros, 3, padding='same', activation='relu'))
    modelo.add(layers.Conv2D(filtros, 3, padding='same', activation='relu'))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.MaxPooling2D())
    modelo.add(layers.Dropout(0.20))

modelo.add(layers.GlobalAveragePooling2D())
modelo.add(layers.Dense(64, activation='relu'))
modelo.add(layers.Dropout(0.30))
modelo.add(layers.Dense(len(CLASES), activation='softmax', name='emociones'))
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
modelo.summary()

# Verificación de entrada/salida; esto aún NO entrena la red.
lote, _ = next(iter(train_ds))
probabilidades = modelo(lote, training=False)
tf.debugging.assert_equal(tf.shape(probabilidades)[1], len(CLASES))
tf.debugging.assert_all_finite(probabilidades, 'Salida no válida')
tf.debugging.assert_near(tf.reduce_sum(probabilidades, axis=1), tf.ones(tf.shape(lote)[0]))
print('Red comprobada. Entrada:', lote.shape, '| salida:', probabilidades.shape)
print('Orden de salida:', dict(enumerate(CLASES)))


Model: "cnn_expresiones"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 48, 48, 1)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 48, 48, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 16)     │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 

## 4. Entrenar
Hasta 30 épocas. Se guarda el modelo con menor pérdida de validación y se detiene si no mejora durante seis épocas. Cada ejecución crea una carpeta distinta. Test no participa en la selección del modelo.


In [ ]:
# Entrenar: ejecutar después de crear el modelo.
from pathlib import Path
from datetime import datetime
import json

SALIDA = BASE / 'modelos' / datetime.now().strftime('CNN_%Y%m%d_%H%M%S_%f')
SALIDA.mkdir(parents=True, exist_ok=False)
(SALIDA / 'clases.json').write_text(json.dumps(CLASES, ensure_ascii=False, indent=2), encoding='utf-8')

# Caché en RAM para evitar releer las imágenes en cada época.
train_fit = train_ds.unbatch().cache().shuffle(
    len(x_train), seed=SEMILLA, reshuffle_each_iteration=True
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_fit = val_ds.cache().prefetch(tf.data.AUTOTUNE)

callbacks = [
    keras.callbacks.ModelCheckpoint(str(SALIDA / 'mejor_modelo.keras'),
        monitor='val_loss', mode='min', save_best_only=True),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6,
        restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
        patience=2, min_lr=0.00001),
    keras.callbacks.CSVLogger(str(SALIDA / 'historial.csv')),
    keras.callbacks.TerminateOnNaN(),
]
print('Archivos del entrenamiento:', SALIDA)
historial = modelo.fit(train_fit, validation_data=val_fit, epochs=EPOCAS, callbacks=callbacks)
# Recuperar exactamente el mejor checkpoint, incluyendo el optimizador.
modelo = keras.models.load_model(SALIDA / 'mejor_modelo.keras')
print('Mejor modelo cargado:', SALIDA / 'mejor_modelo.keras')


(SALIDA / 'configuracion.json').write_text(json.dumps({'clases': CLASES, 'semilla': SEMILLA, 'imagen': [48, 48, 1], 'normalizacion': 'dividir por 255', 'epocas_maximas': EPOCAS, 'tensorflow': tf.__version__}, indent=2), encoding='utf-8')


## 5. Revisar curvas de entrenamiento y validación


In [ ]:
# Ver curvas después de terminar el entrenamiento.
import matplotlib.pyplot as plt

if 'historial' not in globals():
    raise RuntimeError('Primero ejecuta la celda Entrenar y espera a que termine.')

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for eje, metrica, titulo in zip(ejes, ['accuracy', 'loss'], ['Exactitud', 'Pérdida']):
    eje.plot(historial.history[metrica], label='Entrenamiento')
    eje.plot(historial.history['val_' + metrica], label='Validación')
    eje.set(title=titulo, xlabel='Época (índice desde 0)')
    eje.legend()
    eje.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(SALIDA / 'curvas.png', dpi=150)
plt.show()


## 6. Evaluación final
Ejecuta esta sección cuando hayas terminado de elegir el modelo usando validación. No ajustes la red basándote repetidamente en test.


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

if 'SALIDA' not in globals() or not (SALIDA / 'mejor_modelo.keras').is_file():
    raise RuntimeError('Primero termina el entrenamiento.')
modelo_final = keras.models.load_model(SALIDA / 'mejor_modelo.keras')
metricas = modelo_final.evaluate(test_ds, return_dict=True)
predicciones = np.argmax(modelo_final.predict(test_ds, verbose=0), axis=1)
reporte = classification_report(y_test, predicciones, labels=list(range(len(CLASES))),
                                target_names=CLASES, digits=3, zero_division=0)
print(metricas)
print(reporte)
(SALIDA / 'evaluacion_test.json').write_text(json.dumps(metricas, indent=2), encoding='utf-8')
(SALIDA / 'reporte_test.txt').write_text(reporte, encoding='utf-8')
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, predicciones, labels=list(range(len(CLASES))),
    display_labels=CLASES, cmap='Blues', xticks_rotation=45, ax=ax)
fig.tight_layout()
fig.savefig(SALIDA / 'matriz_confusion.png', dpi=150)
plt.show()
